In [ ]:
import sys
sys.path.append('..')

import ast
import pandas as pd
import collections

from config.settings import TmpPaths, SCOPUS_BASE

# Load Climate Modeling cluster data (authors with HIC-Sci flags and EIDs)
policy_cited_scopus2 = pd.read_csv(TmpPaths.BASE / 'climate_modeling_authors.csv', index_col=0)
policy_cited_scopus2['eids'] = policy_cited_scopus2['eids'].apply(ast.literal_eval)

In [9]:
target_eids = policy_cited_scopus2.explode(['eids'])['eids'].unique()

In [11]:
target_eids_hic_sci = policy_cited_scopus2.explode(['eids'])[['eids','is_hic_sci']].reset_index()
target_eids_hic_sci = target_eids_hic_sci[['eids','is_hic_sci']]
target_eids_hic_sci.drop_duplicates('eids', inplace=True)
target_eids_hic_sci.rename(columns={'eids':'eid'},inplace=True)

In [13]:
abstract_df = pd.read_pickle(f'{SCOPUS_BASE}/paper/abstract.pickle')
abstract_df = pd.DataFrame(abstract_df)
abstract_df.reset_index(inplace=True)
abstract_df.rename(columns={'index':'eid'},inplace=True)

In [14]:
title_df = pd.read_pickle(f'{SCOPUS_BASE}/paper/title.pickle')
title_df = pd.DataFrame(title_df)
title_df.reset_index(inplace=True)
title_df.rename(columns={'index':'eid'},inplace=True)

In [15]:
year_df = pd.read_pickle(f'{SCOPUS_BASE}/paper/year.pickle')
year_df = pd.DataFrame(year_df)
year_df.reset_index(inplace=True)
year_df.rename(columns={'index':'eid'},inplace=True)

In [17]:
abstract_df1 = abstract_df[abstract_df['eid'].isin(target_eids)]

In [18]:
title_df1 = title_df[title_df['eid'].isin(target_eids)]

In [19]:
policy_cited_scopus = pd.merge(title_df1, abstract_df1, on='eid', how='left')

In [20]:
policy_cited_scopus['text']= policy_cited_scopus['title'] + ' .'  + policy_cited_scopus['abstract']

In [21]:
policy_cited_scopus = pd.merge(policy_cited_scopus, year_df, on='eid', how='left')

In [22]:
policy_cited_scopus = pd.merge(policy_cited_scopus, target_eids_hic_sci, on='eid', how='left')

In [ ]:
flatten2 = lambda l: [item for sublist in l if sublist == sublist for item in sublist]

In [28]:
import nltk
from nltk.corpus import stopwords
sw = set(stopwords.words('english'))
sw = sw | set(['<span','(<span', '>', "al.", "al.", '°c', '(last', 'idcombining', 'classcombining', 'mathvariantcombining', 'overflowcombining', 'under:', ')',
               'md5hashcombining', 'idcombining', 'heightcombining', 'xlink:hrefcombining', 'xmlnscombining',"(formula",
              "=","(n","presented.)","~","∼", "al.","el","2020","2021","2022","2019","2018","2017","2016","2015","significant","significantly"])

rm = {"al. 2019).", "et al. 2020).", "al. 2020).", "et al. 2019).", "al. 2019)", "1979–2018 period", "2019) al. et", "2019 january", "11 global","2019) al. et"}


def is_symble_(s):
    if 'line"' in s or "xmlns:" in s or "displaycombining" in s or "dspmathcombining" in s:
        return True
    if 'inline-formula' in s:
        return True

def not_symble(b):
    for s in b:
        if is_symble_(s):
            return False
    return True

not_symble(("a",'line","aaa"'))

False

In [31]:

    

def get_trigram(x):
    res = [ [" ".join(b).replace(",","") for b in  list(nltk.trigrams(t.lower().split())) if len(set(b) & set(sw))==0  and not_symble(b)    ]  for t in x.split(". ")]
    return flatten2(res)

def get_bigram(x):
    res = [ [" ".join(b).replace(",","") for b in  list(nltk.bigrams(t.lower().split())) if len(set(b) & set(sw))==0  and not_symble(b)    ]  for t in x.split(". ")]
    return flatten2(res)
    
policy_cited_scopus['keyword'] = policy_cited_scopus['text'].apply(lambda x: get_bigram(x) + get_trigram(x) ).apply(set).apply(lambda x: x-rm)

In [35]:
def remove_duplicates(phrases):
    return list(set(phrases))
from nltk.corpus import wordnet
# Function to find the base form (lemma) of a word using WordNet
def get_lemma(word):
    lemma = wordnet.morphy(word)
    return lemma if lemma else word

# Function to standardize phrases by converting all words to their base forms
def standardize_by_lemma(phrases):
    standardized_phrases = []
    for phrase in phrases:  # phrasesはすでにリスト形式
        words = phrase.lower().split()
        lemma_words = [get_lemma(word) for word in words]
        sorted_lemma_phrase = ' '.join(sorted(lemma_words))
        standardized_phrases.append(sorted_lemma_phrase)
    return standardized_phrases

# 'keyword' 列がリスト形式のデータに対して適用する
policy_cited_scopus['standardized_keyword'] = policy_cited_scopus['keyword'].apply(standardize_by_lemma)

# 重複を除去
policy_cited_scopus['standardized_keyword'] = policy_cited_scopus['standardized_keyword'].apply(remove_duplicates)

In [37]:
year_words_list = policy_cited_scopus.groupby('year')['standardized_keyword'].agg(flatten2)
year_words = year_words_list.apply(set)


In [38]:
word_count = pd.Series(collections.Counter(flatten2(year_words_list)))

In [40]:
target_words = set(word_count[word_count>=5].index)
len(target_words)

30379

In [41]:
total_word = set()
new_word ={}
for k,v in year_words.sort_index().items():
    new_word[k] =   (v - total_word) & target_words
    total_word = total_word | v
    print(len(total_word))

158927
308400
447956
595567
712510
811388
861373
883008
888292


In [43]:
word_count_richclub = pd.concat([pd.Series(collections.Counter(flatten2(policy_cited_scopus.query('is_hic_sci == 1').standardized_keyword) )),
           pd.Series(collections.Counter(flatten2(policy_cited_scopus.standardized_keyword) ))],axis=1).fillna(0)
word_count_richclub.columns = ['RC','total']
word_count_richclub['RC_ratio'] = word_count_richclub.RC / word_count_richclub.total

In [52]:

res = pd.Series(new_word).loc[2018:].explode().to_frame()
res['RC_ratio'] = res[0].map(word_count_richclub['RC_ratio'])
res = res.dropna()
res.rename(columns={0:'standardized_keyword'},inplace=True)
res = res.sort_values('RC_ratio')
res = res[(res['standardized_keyword']!='2019) al. et') & (res['standardized_keyword']!='2019 january')& (res['standardized_keyword']!='2018) al.')]

# Define the conditions for the classificationd
data1 = res

conditions = [
    (data['RC_ratio'] == 0),
    (data['RC_ratio'] > 0) & (data['RC_ratio'] < 0.3),
    (data['RC_ratio'] == 0.3),
    (data['RC_ratio'] > 0.3) & (data['RC_ratio'] < 1),
    (data['RC_ratio'] == 1)
]

# Define the corresponding category labels
category_labels = ['PPR 0', 'PPR 0 to 0.3', 'PPR 0.3', 'RRP 0.3 to 1', 'PPR 1']

# Assign the category labels
data1['category'] = pd.cut(data1['RC_ratio'], bins=[-0.01, 0, 0.3, 0.999, 1], 
                          labels=['PPR 0', 'PPR 0 to 0.3', 'RRP 0.3 to 1', 'PPR 1'], include_lowest=True)

# Group the standardized keywords by their respective categories into lists
categorized_keywords1 = data1.groupby('category')['standardized_keyword'].apply(list).reset_index()

categorized_keywords1

/tmp/ipykernel_3989542/875041947.py:27: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  categorized_keywords1 = data1.groupby('category')['standardized_keyword'].apply(list).reset_index()


category  \
0         PPR 0   
1  PPR 0 to 0.3   
2  RRP 0.3 to 1   
3         PPR 1   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [ ]:
policy_cited_scopus_2018 = policy_cited_scopus[policy_cited_scopus['year']>=2017]
year_words_list = policy_cited_scopus_2018.groupby('year')['standardized_keyword'].agg(flatten2)
word_count = pd.Series(collections.Counter(flatten2(year_words_list)))

categorized_keywords1['standardized_keyword'] = categorized_keywords1['standardized_keyword'].apply(
    lambda word_list: sorted(
        word_list, 
        key=lambda word: word_count.get(word, 0), 
        reverse=True 
    )
)

categorized_keywords1
categorized_keywords1['top_15_keywords'] = categorized_keywords1['standardized_keyword'].apply(lambda x: x[:15])
categorized_keywords1

category  \
0         PPR 0   
1  PPR 0 to 0.3   
2  RRP 0.3 to 1   
3         PPR 1   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       